In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

import wandb
warnings.filterwarnings("ignore")
os.environ["WANDB_SILENT"] = "true"

# ─────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────
DATA_DIR    = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
OPTION_COLS = ["A", "B", "C", "D", "E"]
N_FOLDS     = 5
RANDOM_SEED = 42

# ─────────────────────────────────────────────────────────
# W&B
# ─────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if not WANDB_API_KEY:
    raise ValueError(
        "WANDB_API_KEY not found. "
        "Go to Kaggle notebook -> Add-ons -> Secrets -> New Secret. "
        "Name: WANDB_API_KEY, Value: your key from https://wandb.ai/authorize"
    )
wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="24f2000817-t22026",
    name="tfidf-lr-scratch",
    config=dict(
        n_folds=N_FOLDS,
        tfidf_q_features=60_000,
        tfidf_opt_features=30_000,
        lr_C=4.0,
        random_seed=RANDOM_SEED,
    ),
)

# ─────────────────────────────────────────────────────────
# MAP@3
# ─────────────────────────────────────────────────────────
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

# ─────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

# ─────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# For each row we build 5 (question, option) pairs.
# Each pair gets its own feature vector; label = 1 if correct.
# ─────────────────────────────────────────────────────────
def make_pairs(df, has_labels=True):
    """
    Expand every row into 5 (question, option) pairs.
    Returns:
        texts_q   : question text repeated per pair
        texts_opt : option text per pair
        labels    : 1/0 per pair (only if has_labels)
        row_ids   : original row index per pair (for grouping at inference)
    """
    texts_q, texts_opt, labels, row_ids = [], [], [], []
    for idx, row in df.iterrows():
        q = str(row["prompt"])
        for c in OPTION_COLS:
            texts_q.append(q)
            texts_opt.append(str(row[c]))
            row_ids.append(idx)
            if has_labels:
                labels.append(1 if row["answer"] == c else 0)
    return texts_q, texts_opt, labels, row_ids

train_q, train_opt, train_labels, train_ids = make_pairs(train_df, has_labels=True)
test_q,  test_opt,  _,            test_ids  = make_pairs(test_df,  has_labels=False)

# ─────────────────────────────────────────────────────────
# TF-IDF VECTORISERS  (fit on all text to cover test vocab)
# Two separate vectorisers:
#   1. question TF-IDF
#   2. option   TF-IDF
# Then concatenate horizontally → richer feature space
# ─────────────────────────────────────────────────────────
print("\nFitting TF-IDF vectorisers ...")

tfidf_q = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=60_000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1,
)
tfidf_opt = TfidfVectorizer(
    ngram_range=(1, 3),     # trigrams help for short option phrases
    max_features=30_000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1,
)

# fit on combined train + test text
tfidf_q.fit(train_q + test_q)
tfidf_opt.fit(train_opt + test_opt)

X_train_q   = tfidf_q.transform(train_q)
X_train_opt = tfidf_opt.transform(train_opt)
X_train     = hstack([X_train_q, X_train_opt])   

X_test_q    = tfidf_q.transform(test_q)
X_test_opt  = tfidf_opt.transform(test_opt)
X_test      = hstack([X_test_q, X_test_opt])    

print(f"Feature matrix: train {X_train.shape}  |  test {X_test.shape}")

# ─────────────────────────────────────────────────────────
# HELPER: reshape flat probs → top-3 ranked labels per row
# ─────────────────────────────────────────────────────────
def probs_to_top3(probs_flat, n_questions):
    probs_2d = probs_flat.reshape(n_questions, 5)
    return [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3]
            for r in probs_2d]

# ─────────────────────────────────────────────────────────
# 5-FOLD CROSS VALIDATION
# We train one LR per fold and average test probabilities.
# ─────────────────────────────────────────────────────────
print(f"\n5-Fold Cross Validation ...")

# labels per question (not per pair) for stratified split
question_labels = [row["answer"] for _, row in train_df.iterrows()]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_probs  = np.zeros(len(train_df) * 5)  
test_probs = np.zeros(len(test_df)  * 5) 

fold_scores = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, question_labels)):
    # expand fold indices from question-level → pair-level
    tr_pair_mask = np.isin(train_ids, train_df.index[tr_idx])
    vl_pair_mask = np.isin(train_ids, train_df.index[vl_idx])

    X_tr = X_train[tr_pair_mask]
    y_tr = np.array(train_labels)[tr_pair_mask]
    X_vl = X_train[vl_pair_mask]
    y_vl = np.array(train_labels)[vl_pair_mask]

    clf = LogisticRegression(
        C=4.0,
        max_iter=1000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )
    clf.fit(X_tr, y_tr)

    # val probs
    vl_probs = clf.predict_proba(X_vl)[:, 1]
    oof_probs[vl_pair_mask] = vl_probs

    n_val_q  = len(vl_idx)
    val_preds = probs_to_top3(vl_probs, n_val_q)
    val_answers = train_df.iloc[vl_idx]["answer"].tolist()
    score = mapk(val_answers, val_preds)
    fold_scores.append(score)
    print(f"  Fold {fold+1} val MAP@3 = {score:.4f}")
    wandb.log({"fold": fold + 1, "val/map3": score})

    # accumulate test probs
    test_probs += clf.predict_proba(X_test)[:, 1]

test_probs /= N_FOLDS   # average across folds

print(f"\nCV MAP@3 per fold : {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean CV MAP@3     : {np.mean(fold_scores):.4f}")

# W&B CV summary
wandb.run.summary["cv_map3_mean"] = float(np.mean(fold_scores))
wandb.run.summary["cv_map3_std"]  = float(np.std(fold_scores))
wandb.log({
    "cv_fold_map3": wandb.plot.bar(
        wandb.Table(
            columns=["fold", "val_map3"],
            data=[[f"fold{i+1}", s] for i, s in enumerate(fold_scores)],
        ),
        "fold", "val_map3", title="Val MAP@3 per Fold",
    )
})

# OOF score
oof_preds   = probs_to_top3(oof_probs, len(train_df))
oof_map3    = mapk(train_df["answer"].tolist(), oof_preds)
print(f"OOF MAP@3         : {oof_map3:.4f}")
wandb.run.summary["oof_map3"] = float(oof_map3)

# ─────────────────────────────────────────────────────────
# SUBMISSION
# ─────────────────────────────────────────────────────────
test_preds = probs_to_top3(test_probs, len(test_df))

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
out_path = "/kaggle/working/submission_scratch.csv"
submission.to_csv(out_path, index=False)
print(f"\n✅  Saved -> {out_path}")
wandb.finish()
print(submission.head(10).to_string(index=False))

Train: 2000  |  Test: 500

Fitting TF-IDF vectorisers ...
Feature matrix: train (10000, 25351)  |  test (2500, 25351)

5-Fold Cross Validation ...
  Fold 1 val MAP@3 = 0.9838
  Fold 2 val MAP@3 = 0.9950
  Fold 3 val MAP@3 = 0.9912
  Fold 4 val MAP@3 = 0.9875
  Fold 5 val MAP@3 = 0.9925

CV MAP@3 per fold : ['0.9838', '0.9950', '0.9912', '0.9875', '0.9925']
Mean CV MAP@3     : 0.9900
OOF MAP@3         : 0.9900

✅  Saved -> /kaggle/working/submission_scratch.csv
 ID Prediction
  1      A E B
  2      B E D
  3      B E D
  4      E C A
  5      C A D
  6      D A B
  7      E D A
  8      B E A
  9      C D E
 10      B C D
